# 02 — Build an inspectable local RAG assistant

**Level:** Beginner · **Estimated time:** 75–90 minutes · **Scenario:** Harborline Support

By the end, you will build, run, debug, and evaluate a dependency-free local RAG assistant. You will explain every score it produces, audit its corpus, build a bounded context window, return citations, and make abstention a measurable product behavior.

> The goal is not to make a clever chatbot. It is to establish a baseline you can falsify before adding embeddings or an LLM.


## How to use this notebook

Work in this order: read the concept, run the deterministic code, change **one** variable, inspect the trace, and write down what changed. The model API is deliberately absent: the learning objective is to understand the evidence system that an LLM would depend on.

**Scenario.** You are building a small, internal assistant for Harborline, a fictional SaaS company. Support needs trustworthy answers about customer communication and production escalation. The corpus is intentionally tiny so every result can be inspected.


## 1. What RAG changes — and what it does not

Retrieval-augmented generation (RAG) supplies external evidence at answer time. It does **not** make an answer automatically correct: ingestion can omit a document, chunking can split an answer, retrieval can rank the wrong evidence, and generation can overstate what evidence supports.

```text
OFFLINE:  documents → parse → chunks + metadata → index

ONLINE:   question → retrieve candidates → threshold policy
                                            ├─ enough evidence → bounded context → answer + citations
                                            └─ weak / empty    → abstain + next safe step
```

Keep two questions separate throughout this course:

1. **Retrieval quality:** did the system find the evidence needed to answer?
2. **Answer faithfulness:** did the final answer stay within that evidence?

This notebook evaluates the first question and implements a simple evidence boundary. The citation notebook later adds stronger provenance checks.


## 2. Why start with lexical retrieval?

Lexical retrieval ranks literal term overlap. It is weak on synonyms, but it makes failure visible: a learner can see that `payment incident` matched one source while `refund outage` did not. Dense retrieval can improve paraphrase matching later; it does not remove the need for evaluation, metadata, or an abstention policy.

The baseline has four contracts:

- **Stable identity:** every chunk has an ID and source filename.
- **Inspectable ranking:** every hit includes score, rank, and matched terms.
- **Bounded context:** only a limited amount of labelled evidence reaches the answer step.
- **Safe terminal state:** insufficient evidence returns an abstention rather than invented support.


In [ ]:
from pathlib import Path
from examples.beginner.first_local_rag import (
    EvaluationCase, build_context, evaluate_baseline, load_chunks,
    retrieve_with_trace,
)

ROOT = Path.cwd() if (Path.cwd() / 'examples').exists() else Path('../..')
DOCS = ROOT / 'examples/data/beginner-docs'
chunks = load_chunks(DOCS)
print(f'Indexed {len(chunks)} inspectable chunks from {len({c.source for c in chunks})} documents.')
[(chunk.chunk_id, chunk.source, chunk.section) for chunk in chunks[:8]]


## 3. Inspect ingestion before you trust retrieval

Ingestion is a correctness boundary. If a required policy is absent or a source has no stable ID, an apparently good answer cannot be audited. Our loader intentionally uses Markdown paragraphs as chunks. That is not a production recommendation; it is a simple unit that lets us inspect the pipeline end to end.

**Check before continuing:** locate the `Harborline Support Handbook` and its `Escalation boundary` paragraph. What information would you add before using this data in production (for example document version, tenant, access label, or updated timestamp)?


In [ ]:
question = 'Who may restart production services?'
hits = retrieve_with_trace(question, chunks, top_k=3)

for hit in hits:
    print(f'#{hit.rank} score={hit.score:.2f} matches={hit.matched_terms}')
    print(f'   {hit.chunk.chunk_id} · {hit.chunk.source} · {hit.chunk.section}')
    print(f'   {hit.chunk.text}\n')


## 4. Read a retrieval trace

The score is the fraction of query terms found in a chunk. It is **not** a calibrated probability of truth. A score of `0.60` means three of five lexical terms overlapped after stop words were removed. It says nothing about whether the passage authorizes an action or whether the text is current.

Useful debugging questions:

- Which important query term failed to match?
- Is the first hit actually responsive, or merely keyword-adjacent?
- Did a heading become a low-value retrieval candidate?
- Would an alternate phrasing retrieve a different source?


In [ ]:
paraphrase = 'Can the support team reboot checkout in production?'
for prompt in (question, paraphrase):
    trace = retrieve_with_trace(prompt, chunks, top_k=3)
    print(f'\n{prompt}')
    print([(hit.chunk.chunk_id, round(hit.score, 2), hit.matched_terms) for hit in trace])


## 5. Build context, not a document dump

Retrieval is normally followed by prompt construction. Context should be labelled, bounded, and ordered; otherwise a large or irrelevant document can hide the useful evidence, inflate latency, and make citations ambiguous.

```text
ranked hits → label with stable source IDs → apply a context budget → answer from the retained evidence
```

In a real system, the context builder would additionally apply authorization before retrieval, preserve versions and locations, and avoid placing untrusted instructions in a trusted prompt channel.


In [ ]:
context = build_context(hits, max_characters=420)
print(context)
print('\nCharacters retained:', len(context))


## 6. The abstention policy is a product decision

A baseline needs a documented answer/no-answer boundary. A weak score should not be converted into polished prose. At the same time, a threshold that is too high can hide a valid answer. Treat the threshold as a policy parameter and test it on both supported and unsupported questions.

**Failure case:** “What is the capital of France?” is outside the Harborline corpus. It should abstain even though a general-purpose model might know an answer.


In [ ]:
from examples.beginner.first_local_rag import answer

for prompt in (question, 'What is the capital of France?'):
    print(f'Q: {prompt}')
    print(answer(prompt, chunks, min_score=0.20), '\n')


## 7. Evaluate retrieval and abstention separately

A **golden set** names representative questions, the evidence IDs expected for answerable questions, and which questions should abstain. It turns a demo into an engineering baseline. Notice that retrieval success and abstention correctness can disagree: a good retriever can still use a badly tuned threshold.


In [ ]:
golden_set = [
    EvaluationCase('Who may restart production services?', ('harborline-support-7',)),
    EvaluationCase('How often do enterprise customers receive an update?', ('harborline-support-5',)),
    EvaluationCase('What must an answer distinguish?', ('harborline-policy-3',)),
    EvaluationCase('What is the capital of France?', (), should_abstain=True),
]

report = evaluate_baseline(golden_set, chunks, top_k=3, min_score=0.20)
for row in report:
    print(row)

retrieval_recall = sum(row['retrieval_hit'] for row in report if not row['abstention_correct'] or row['retrieval_hit']) / len(report)
abstention_accuracy = sum(row['abstention_correct'] for row in report) / len(report)
print(f'\nIllustrative retrieval-hit rate: {retrieval_recall:.0%}')
print(f'Abstention accuracy: {abstention_accuracy:.0%}')


## 8. Experiment: change one variable

Run the same golden set at `top_k=1` and `top_k=3`, then at thresholds `0.20`, `0.50`, and `0.80`. Do not change both at once.

Record:

| Change | Retrieval hit? | Abstention correct? | What became worse? |
| --- | --- | --- | --- |
| `top_k=1` | | | |
| `min_score=0.50` | | | |
| your chosen policy | | | |

**Interpretation:** choose a policy from representative failures, not from one impressive answer.


In [ ]:
# Your experiment: alter only one parameter in each run.
for threshold in (0.20, 0.50, 0.80):
    rows = evaluate_baseline(golden_set, chunks, top_k=3, min_score=threshold)
    correct = sum(row['abstention_correct'] for row in rows)
    print(f'threshold={threshold:.2f} · abstention accuracy={correct}/{len(rows)}')


## 9. Deliberate failure: lexical mismatch

Ask a question using words absent from the source, such as “Can support **reboot** checkout?” while the policy says “restart production services.” If the baseline misses it, that is expected evidence for a later change—not a reason to hide the failure.

Possible next interventions, in order of increasing complexity:

1. improve source wording or add synonyms where editorially correct;
2. add query expansion and evaluate whether it drifts from intent;
3. add dense or hybrid retrieval, retaining the same golden set;
4. rerank a bounded candidate set.

Do not adopt a technique just because it returns an answer; verify that it retrieves the right evidence and preserves citations.


In [ ]:
failure_prompt = 'Can support reboot checkout?'
[(hit.chunk.chunk_id, hit.score, hit.matched_terms) for hit in retrieve_with_trace(failure_prompt, chunks)]


## 10. Audit the corpus before you index it

An answer can only be as auditable as the source record that produced it. Before experimenting with rankers, verify that the corpus has stable IDs, non-empty content, and an understandable document/chunk count. This small audit does not prove freshness or authorization; it proves that basic ingestion defects are visible.

For production, add source owner, version/hash, update time, tenant/ACL, retention state, extraction method, and index version. Keep those fields with the retrieval record rather than asking a model to remember them.


In [ ]:
from examples.beginner.first_local_rag import audit_corpus

audit = audit_corpus(chunks)
print(audit)
assert audit.ready, 'Fix corpus defects before testing retrieval.'


## 11. Run the local pipeline as one auditable operation

A useful local application returns more than a string. It should preserve the query, retrieval threshold, top hits, bounded context, citations, and the terminal decision. `run_local_rag` models that hand-off without making a network call. In a provider-backed version, replace only the final deterministic rendering step; retain the evidence and policy contract.


In [ ]:
from examples.beginner.first_local_rag import run_local_rag

result = run_local_rag(
    'How should support communicate during a confirmed payment incident?',
    chunks,
    top_k=3,
    min_score=0.20,
    max_characters=520,
)
print('decision:', result.decision)
print('trace:', [(hit.rank, hit.chunk.chunk_id, round(hit.score, 2)) for hit in result.hits])
print('citations:', result.citations)
print('\n' + result.answer)


## 12. Context is a constrained evidence package

Putting every retrieved document into a prompt makes citations ambiguous and hides the most relevant rule. The context pack retains labels and records when a character budget excludes lower-ranked evidence. A tight budget may be correct for a short answer, but it can be dangerous if it drops a rule exception.

The policy question is not `how much context can the model hold?`; it is `which evidence is necessary and sufficient for this claim?`


In [ ]:
from examples.beginner.first_local_rag import build_context_pack

pack = build_context_pack(result.hits, max_characters=260)
print('retained IDs:', pack.retained_ids)
print('truncated:', pack.truncated)
print('citations:', pack.citations)
print('\n' + pack.text)


## 13. Compare lexical overlap and BM25

The overlap baseline assigns every query term the same value. BM25 additionally rewards discriminative terms and normalizes for document length. It remains lexical: it can make keyword search a stronger baseline, but it cannot necessarily understand `reboot` as `restart`.

Sentence Transformers can later encode a short query and longer corpus passage for asymmetric semantic search; Qdrant and similar databases can store vectors with payload metadata for filtered search. Adopt those tools only after a golden-set failure says what they should improve.


In [ ]:
from examples.beginner.first_local_rag import retrieve_bm25

for prompt in (
    'How often do enterprise customers receive an update?',
    'Can support reboot checkout in production?',
):
    overlap = [(h.chunk.chunk_id, round(h.score, 3)) for h in retrieve_with_trace(prompt, chunks)]
    bm25 = [(h.chunk.chunk_id, round(h.score, 3)) for h in retrieve_bm25(prompt, chunks)]
    print(f'\n{prompt}')
    print('overlap:', overlap)
    print('BM25:   ', bm25)


## 14. Tune the decision policy on a golden set

A good demo is not evidence of a good system. A golden set contains answerable, unsupported, paraphrased, ambiguous, and later permission-restricted questions. For each case, record expected chunk IDs and expected terminal behavior.

Run this experiment by varying only one parameter at a time. A lower threshold may reduce false abstentions while allowing weak evidence; a larger `top_k` can improve recall while making context noisier. Choose a policy from the cost of these errors for Harborline support, not from one attractive answer.


In [ ]:
for top_k, threshold, budget in ((1, 0.20, 520), (3, 0.20, 520), (3, 0.50, 260)):
    rows = evaluate_baseline(golden_set, chunks, top_k=top_k, min_score=threshold)
    retrieval_hits = sum(row['retrieval_hit'] for row in rows)
    abstentions = sum(row['abstention_correct'] for row in rows)
    print(
        f'top_k={top_k}, threshold={threshold:.2f}, context_budget={budget}: '
        f'retrieval hits={retrieval_hits}/{len(rows)}, abstentions correct={abstentions}/{len(rows)}'
    )


## 15. Debug at the boundary that failed

```text
missing canonical text? -> source ownership / ingestion / freshness
text present but no hit?  -> chunking / query terms / ranker
good hit, bad answer?     -> context selection / generation constraint
citation but old text?    -> versioning / re-indexing policy
restricted text visible?  -> authorization before retrieval
no evidence available?    -> abstain + safe escalation
```

Do not use a larger model to cover a source, access-control, or index-freshness defect. A typed, authenticated API is often a better solution for live account values or production actions.


## 16. Build exercise and checkpoint

1. Add a fourth Harborline document with a version and an exception rule. Explain whether the paragraph chunk preserves the exception.
2. Create five golden cases: supported, unsupported, synonym mismatch, ambiguous, and stale-source cases.
3. Set `max_characters=180`. Which context IDs remain, and can the answer still be supported?
4. Sketch the production record you would store for every chunk: source/version, ACL/tenant, index version, retrieval score, and citation location.
5. Which observed failure would justify semantic retrieval? Which one would make an authenticated API a safer choice?

**Next:** the [Chunking lab](../../curriculum/beginner/03-chunking-lab/README.md) changes the unit retrieval can return and measures the effect.

### References

- Lewis et al., [Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks](https://arxiv.org/abs/2005.11401)
- Manning, Raghavan, and Schütze, [Introduction to Information Retrieval](https://nlp.stanford.edu/IR-book/)
- Thakur et al., [BEIR](https://arxiv.org/abs/2104.08663)
- Sentence Transformers, [Semantic Search](https://www.sbert.net/examples/sentence_transformer/applications/semantic-search/README.html)
- Qdrant, [Overview](https://qdrant.tech/documentation/overview/)
